# Impersonation Worker (English): Training Notebook
**What this does:** trains a small model that reads a few lines of a phone call and gives a score from 0 to 1 for
"the caller is falsely claiming to be an authority or trusted organisation" (police, bank, government, telco, courier, tech support).

**Data files:** `impersonation_data.csv` (training rows) and `gold_test.csv` (the final test lines, never used for tuning).
Required columns: `text`, `label` (1 or 0). Optional columns: `source`, `scenario_id`, `org_category`, `is_hard_negative`, `evidence`, `style`.
- `source` says where a row came from (for example `synthetic_advisory_based`, `hand_written`). The notebook reports results per source and for synthetic versus non-synthetic rows.
- `scenario_id` is any name shared by all variations of the same script, so they stay on the same side of every split.
- `is_hard_negative` is 1 for genuine calls from a real organisation.
- Rows whose label is not 0 or 1 (for example `U` for unclear) are dropped automatically.

## 1. Install libraries

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "pandas", "numpy", "matplotlib", "joblib"], check=False)
print("Libraries ready")

## 2. Settings (the only cell you normally need to edit)
- `NGRAM_RANGE`: (1, 2) means single words and two-word pairs. Try (1, 3) to include three-word phrases.
- `MIN_DF`: ignore words/phrases that appear in fewer than this many examples. Higher = less memorising.
- `MAX_FEATURES`: the most words/phrases the model may use.
- `C`: regularisation strength. Smaller = simpler model. The notebook also tries several other values and keeps the best by cross-validation.
- `THRESHOLD`: scores at or above this count as impersonation when we print precision/recall. Your teammate's aggregator uses the raw score.
- `CALLER_ONLY`: True = the model only reads what the caller says (the receiver's turns are dropped). The same setting is baked into the exported model.
- `MASK_BRANDS`: True = replace org names (SPF, DBS...) with their type (orgpolice, orgbank...) so the model learns "claims to be a bank/police/government body" instead of memorising brands.
- `TRY_CHAR_NGRAMS`: True = also try adding character n-grams (helps with typos and speech-to-text errors). The notebook keeps whichever scores better in cross-validation.
- `CALIBRATE`: True = adjust scores so 0.8 really means "about 80% likely" (the aggregator needs this). Calibration uses the out-of-fold scores from cross-validation.
- `AUTO_THRESHOLD`: True = pick the reporting threshold from the out-of-fold scores (never from the test or gold set). False = use `THRESHOLD`.
- `N_FOLDS`: how many cross-validation folds. Every fold keeps whole scripts together. More folds = steadier choices, slower run.
- `NOISE_COPIES`: how many noisy copies (dropped words, typos, "um") of each training row to add, to mimic speech-to-text. 0 = none. Noisy copies are never added to the rows being scored.
- `BOOTSTRAPS`: how many resamples for the confidence ranges. Higher = smoother, slower.
- `NEAR_DUP_THRESHOLD`: two examples this similar (0 to 1) are treated as the same script and kept in the same split. Also used to warn if a gold line is nearly identical to a training line.
- `SYNTHETIC_PATTERN`: a `source` value matching this text is counted as synthetic (AI-written). Everything else is reported as non-synthetic.
- `NONCOMMERCIAL_SOURCE_MARKERS`: if a `source` contains one of these words, the notebook warns that the data has a non-commercial licence.

In [ ]:
DATA_CSV = "impersonation_data.csv"
GOLD_CSV = "gold_test.csv"

NGRAM_RANGE = (1, 2)
MIN_DF = 2
MAX_FEATURES = 30000
C = 1.0
THRESHOLD = 0.5
CALLER_ONLY = True
MASK_BRANDS = True
NEAR_DUP_THRESHOLD = 0.8
TRY_CHAR_NGRAMS = True
CALIBRATE = True
AUTO_THRESHOLD = True
NOISE_COPIES = 1
BOOTSTRAPS = 1000
N_FOLDS = 5

SYNTHETIC_PATTERN = "synthetic|ai_written|llm"
NONCOMMERCIAL_SOURCE_MARKERS = ("92k", "call_center", "callcenter", "cc-by-nc")

RANDOM_STATE = 42
TEST_SIZE = 0.15
WINDOW_TURNS = 3
MIN_PER_CLASS = 5
MODEL_VERSION = "impersonation_en_v2"

In [ ]:
import os, re, json, time, warnings
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import Pipeline, FeatureUnion
import random
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from scipy.special import expit
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score, average_precision_score,
                             brier_score_loss, confusion_matrix, precision_recall_curve)
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
os.makedirs("outputs", exist_ok=True)

## 3. Text cleaning helpers
These are saved into their own file `text_utils.py`. This matters: the model needs this file again when it runs inside Docker.
They keep only the caller's words (every caller turn, nothing cut off), then remove speaker labels (like "Caller:"), redaction tags (like [ORGANIZATION]) and long numbers, so the model cannot cheat on dataset artefacts.
If a text has no speaker labels (for example the 92k call-center set), it is kept whole.

In [ ]:
TEXT_UTILS_SOURCE = '''
import re

TAG_RE = re.compile(r"\\[[A-Za-z_ ]{2,30}\\]")
SPEAKER_RE = re.compile(r"\\b(?:caller|receiver|agent|customer|person\\s*[ab12]|speaker\\s*\\d|scammer|victim|user)\\s*:", re.I)
DIGITS_RE = re.compile(r"\\d{4,}")
SPACE_RE = re.compile(r"\\s+")

# Organisation names are replaced by their TYPE (police, government, bank...), so the model learns "claims to be a bank/police/government body"
# without memorising individual brands like SPF or DBS.
ORG_TYPES = {
    "orgpolice": ["spf", "singapore police force", "interpol"],
    "orggovt": ["ica", "mom", "iras", "mas", "cpf", "singpass", "irs", "immigration and checkpoints authority", "monetary authority of singapore",
                "ministry of manpower", "inland revenue authority of singapore", "cyber security agency", "cpf board", "state courts",
                "supreme court", "ministry of home affairs", "ministry of education", "hdb", "lta", "csa", "govtech",
                "moh", "ministry of health", "town council", "singapore customs", "mha"],
    "orgbank": ["dbs", "ocbc", "uob", "posb", "citibank", "hsbc", "standard chartered", "standard chartered bank", "maybank"],
    "orgtelco": ["singtel", "starhub", "m1"],
    "orgtech": ["microsoft", "apple", "google", "amazon", "paypal"],
    "orgcourier": ["singpost", "dhl", "fedex", "ninja van"],
    "orgshop": ["lazada", "shopee", "grab", "gojek"],
    "orgutility": ["sp group", "sp services"],
    "orghealth": ["sgh", "singapore general hospital", "changi general hospital", "ttsh", "tan tock seng hospital", "nuh", "khoo teck puat hospital", "synapxe"],
}
ORG_NAMES = [n for names in ORG_TYPES.values() for n in names]
NAME_TO_TYPE = {n: t for t, names in ORG_TYPES.items() for n in names}
ORG_RE = re.compile(r"\\b(" + "|".join(re.escape(o) for o in sorted(ORG_NAMES, key=len, reverse=True)) + r")\\b")


def clean_text(t):
    t = str(t)
    t = TAG_RE.sub(" ", t)
    t = SPEAKER_RE.sub(" ", t)
    t = t.lower()
    t = DIGITS_RE.sub("longnumber", t)
    return SPACE_RE.sub(" ", t).strip()


def mask_brands(t):
    return ORG_RE.sub(lambda m: NAME_TO_TYPE[m.group(1)], t)


# Speaker labels. The "caller" side is the remote party whose identity we judge.
REMOTE_LABELS = {"caller", "scammer", "agent"}
LOCAL_LABELS = {"receiver", "customer", "victim", "user"}
LABEL_SPLIT_RE = re.compile(r"\\b(caller|scammer|agent|receiver|customer|victim|user)\\s*:", re.I)


def caller_only(text):
    """Keep every word the caller says and drop the receiver's turns.
    If the text has no recognised speaker labels, return it unchanged (nothing is cut)."""
    text = str(text)
    parts = LABEL_SPLIT_RE.split(text)
    if len(parts) == 1:
        return text
    kept = [parts[0]]
    for i in range(1, len(parts), 2):
        if parts[i].lower() in REMOTE_LABELS:
            kept.append(parts[i + 1])
    return " ".join(kept)


def prep(texts, mask=True, caller=True):
    out = []
    for t in texts:
        if caller:
            t = caller_only(t)
        t = clean_text(t)
        if mask:
            t = mask_brands(t)
        out.append(t)
    return out


def preprocess(texts):
    return prep(texts, mask=False, caller=True)


def preprocess_masked(texts):
    return prep(texts, mask=True, caller=True)


class CalibratedModel:
    """A fitted pipeline plus a small Platt-scaling step, so the 0 to 1 score behaves more like a real probability."""

    def __init__(self, base, calibrator):
        self.base = base
        self.calibrator = calibrator

    def predict_proba(self, X):
        z = self.base.decision_function(X).reshape(-1, 1)
        return self.calibrator.predict_proba(z)
'''
with open("text_utils.py", "w") as f:
    f.write(TEXT_UTILS_SOURCE)
import importlib, text_utils
importlib.reload(text_utils)
print(text_utils.clean_text("Caller: This is [ORGANIZATION] calling about 1234567 now"))

## 4. Load data
Both files must be uploaded (folder icon on the left in Colab). Rows with a label other than 0 or 1 are dropped.
Rows are marked synthetic or non-synthetic from their `source` value, and every later table reports them separately.

In [ ]:
def ensure_files(paths):
    missing = [p for p in paths if not os.path.exists(p)]
    if missing and "google.colab" in sys.modules:
        from google.colab import files
        print("Upload:", ", ".join(missing))
        files.upload()
        missing = [p for p in paths if not os.path.exists(p)]
    return missing

missing = ensure_files([DATA_CSV, GOLD_CSV])
if DATA_CSV in missing:
    raise FileNotFoundError(f"{DATA_CSV} not found. Upload it with the folder icon on the left.")
df = pd.read_csv(DATA_CSV)
gold = pd.read_csv(GOLD_CSV) if GOLD_CSV not in missing else None
if gold is None:
    print(f"No {GOLD_CSV} found: the gold evaluation will be skipped until you add it.")

def tidy(frame):
    frame = frame.copy()
    for col, default in [("source", "unknown"), ("scenario_id", ""), ("org_category", ""), ("is_hard_negative", 0), ("style", "")]:
        if col not in frame.columns:
            frame[col] = default
    frame = frame.dropna(subset=["text", "label"])
    frame["label"] = pd.to_numeric(frame["label"], errors="coerce")
    dropped = int(frame["label"].isna().sum() + (~frame["label"].isin([0, 1]) & frame["label"].notna()).sum())
    frame = frame[frame["label"].isin([0, 1])].copy()
    frame["label"] = frame["label"].astype(int)
    frame["is_hard_negative"] = pd.to_numeric(frame["is_hard_negative"], errors="coerce").fillna(0).astype(int)
    frame["source"] = frame["source"].fillna("unknown").astype(str)
    frame["is_synthetic"] = frame["source"].str.contains(SYNTHETIC_PATTERN, case=False, regex=True)
    return frame, dropped

df, dropped = tidy(df)
if dropped:
    print(f"Dropped {dropped} rows whose label was not 0 or 1 (for example U for unclear)")
df["clean"] = text_utils.prep(df["text"], mask=False, caller=CALLER_ONLY)
before = len(df)
df = df.drop_duplicates(subset="clean").reset_index(drop=True)
print(f"Rows: {before} -> {len(df)} after removing exact duplicates")
print(df["label"].value_counts().rename({1: "impersonation (1)", 0: "not impersonation (0)"}))
print(df.groupby(["source", "label"]).size())
print(f"Synthetic rows: {int(df.is_synthetic.sum())}   non-synthetic rows: {int((~df.is_synthetic).sum())}")
hn = df.loc[df.label == 0, "is_hard_negative"].mean()
print(f"Hard negatives are {hn:.0%} of the negatives (aim for 40% or more)")
if hn < 0.4:
    print("WARNING: add more hard negatives (legit bank, telco, courier, government calls).")
if (~df.is_synthetic).sum() == 0:
    print("NOTE: every training row is synthetic, so no result below says anything about real calls.")

## 5. Split the data safely
Variations of the same scam script must stay together, otherwise the test score is fake-good.
We group rows with the same `scenario_id`, and also rows that are almost identical (similarity above `NEAR_DUP_THRESHOLD`).
Then the data is split by group, never by single row:
- **Test set** (about 15%): held out and only used to report results.
- **Development set** (the rest): used for cross-validation. Each fold trains on part of it and scores the part it did not see. Choices such as `C`, character n-grams, calibration and the threshold all come from these out-of-fold scores.

In [ ]:
def build_groups(texts, scenario_ids, thr):
    n = len(texts)
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    first = {}
    for i, s in enumerate(scenario_ids):
        s = "" if pd.isna(s) else str(s)
        if s:
            if s in first:
                union(i, first[s])
            else:
                first[s] = i
    X = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit_transform(texts)
    for s in range(0, n, 500):
        sims = (X[s:s + 500] @ X.T).tocsr()
        sims.data[sims.data < thr] = 0
        sims.eliminate_zeros()
        coo = sims.tocoo()
        for r, c in zip(coo.row, coo.col):
            if c > s + r:
                union(s + r, c)
    return np.array([find(i) for i in range(n)])

groups = build_groups(df["clean"].tolist(), df["scenario_id"].tolist(), NEAR_DUP_THRESHOLD)
sizes = pd.Series(groups).value_counts()
print(f"{len(sizes)} script groups. Largest group = {sizes.iloc[0] / len(df):.0%} of data")
if sizes.iloc[0] / len(df) > 0.3:
    print("WARNING: one group is over 30% of the data. Raise NEAR_DUP_THRESHOLD or give rows more specific scenario_id values.")


def make_test_split(df, groups, test_size, seed0):
    y = df["label"].values
    for attempt in range(50):
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed0 + attempt)
        dev_i, test_i = next(gss.split(df, y, groups))
        if all((y[idx] == 1).sum() >= MIN_PER_CLASS and (y[idx] == 0).sum() >= MIN_PER_CLASS for idx in (dev_i, test_i)):
            return dev_i, test_i
    raise RuntimeError("Could not make a split with enough examples of both classes. Add more data.")

dev_idx, te_idx = make_test_split(df, groups, TEST_SIZE, RANDOM_STATE)
dev = df.iloc[dev_idx].reset_index(drop=True)
test = df.iloc[te_idx].reset_index(drop=True)
dev_groups, test_groups = groups[dev_idx], groups[te_idx]

folds = None
for attempt in range(50):
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE + attempt)
    cand = list(sgkf.split(dev, dev["label"], dev_groups))
    if all(dev["label"].values[a].min() != dev["label"].values[a].max() and dev["label"].values[b].min() != dev["label"].values[b].max() for a, b in cand):
        folds = cand
        break
if folds is None:
    raise RuntimeError("Could not make cross-validation folds with both classes in every fold. Add more data or lower N_FOLDS.")
print(f"development {len(dev):5d} rows, {int(dev.label.sum()):4d} impersonation  ({N_FOLDS} cross-validation folds)")
print(f"test        {len(test):5d} rows, {int(test.label.sum()):4d} impersonation")

## 6. Build and train the model
The model has three steps: clean the text, turn it into TF-IDF numbers (word and word-pair importance), then Logistic Regression gives a probability.
- Noisy copies of the training rows are added so the model copes with speech-to-text errors. They are only added to the rows a fold trains on, never to the rows it is scored on.
- Every combination of `C` and word-only versus word + character n-grams is scored by cross-validation (PR-AUC averaged over folds). The best one wins, and ties go to the simplest option.
- The out-of-fold scores (each row scored by a model that never saw it) are then used to calibrate the scores and to choose the reporting threshold.
- The final model is trained on the whole development set. The test and gold sets are never used for any choice.

In [ ]:
def make_pipeline(c, mask, char=False):
    prep = partial(text_utils.prep, mask=mask, caller=CALLER_ONLY)  # a partial of a module-level function can be saved with joblib
    word = TfidfVectorizer(ngram_range=NGRAM_RANGE, min_df=MIN_DF, max_features=MAX_FEATURES, sublinear_tf=True)
    if char:
        feats = FeatureUnion([("word", word),
                              ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=MIN_DF, max_features=MAX_FEATURES, sublinear_tf=True))])
    else:
        feats = word
    return Pipeline([("prep", FunctionTransformer(prep, validate=False)), ("tfidf", feats),
                     ("clf", LogisticRegression(C=c, max_iter=2000, random_state=RANDOM_STATE))])

def add_noise(text, rng):
    """Mimic speech-to-text: drop a few words, add fillers, swap letters in a few words."""
    out = []
    for w in str(text).split():
        r = rng.random()
        if r < 0.04:
            continue
        if r < 0.07:
            out.append(rng.choice(["um", "uh", "like", "you know"]))
        if r < 0.10 and len(w) > 4:
            i = rng.randrange(1, len(w) - 2)
            w = w[:i] + w[i + 1] + w[i] + w[i + 2:]
        out.append(w)
    return " ".join(out) if out else str(text)

def noisy_copy(frame, seed):
    rng = random.Random(seed)
    cleaned = text_utils.prep(frame["text"], mask=False, caller=CALLER_ONLY)  # speaker labels are removed first, then noise is added
    copy = frame.copy()
    copy["text"] = [add_noise(t, rng) for t in cleaned]
    return copy

def with_noise(frame, seed0):
    return pd.concat([frame] + [noisy_copy(frame, seed0 + k) for k in range(NOISE_COPIES)])

def fit_config(frame, c, char, seed0=RANDOM_STATE):
    aug = with_noise(frame, seed0)
    return make_pipeline(c, MASK_BRANDS, char).fit(aug["text"], aug["label"])

C_GRID = sorted({0.1, 0.3, 1, 3, 10, 30, C})
configs = [(char, c) for char in ([False, True] if TRY_CHAR_NGRAMS else [False]) for c in C_GRID]
y_dev = dev["label"].values
oof, rows = {}, []
for char, c in configs:
    z = np.zeros(len(dev))
    fold_ap = []
    for k, (tr, va) in enumerate(folds):
        m = fit_config(dev.iloc[tr], c, char, RANDOM_STATE + 100 * k)
        z[va] = m.decision_function(dev.iloc[va]["text"])
        fold_ap.append(average_precision_score(y_dev[va], z[va]))
    oof[(char, c)] = z
    rows.append((char, c, np.mean(fold_ap), np.std(fold_ap), average_precision_score(y_dev, z)))
tune = pd.DataFrame(rows, columns=["char_ngrams", "C", "mean_fold_PR_AUC", "std_across_folds", "pooled_PR_AUC"])
print(f"Cross-validated tuning on {len(dev)} development rows, {N_FOLDS} folds (test and gold sets not used)")
print(tune.round(4).to_string(index=False))
tune["rounded"] = tune["mean_fold_PR_AUC"].round(3)
best = tune.sort_values("rounded", ascending=False, kind="stable").iloc[0]
best_c, best_char = float(best["C"]), bool(best["char_ngrams"])
print(f"Chosen: C={best_c}, character n-grams={best_char} (ties go to the simplest option)")
z_oof = oof[(best_char, best_c)]

base = fit_config(dev, best_c, best_char)
if CALIBRATE:
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(z_oof.reshape(-1, 1), y_dev)
    model = text_utils.CalibratedModel(base, platt)
    oof_prob = platt.predict_proba(z_oof.reshape(-1, 1))[:, 1]
    print("Scores calibrated on the out-of-fold scores")
else:
    model = base
    oof_prob = expit(z_oof)

if AUTO_THRESHOLD:
    def f1_at(t):
        return precision_recall_fscore_support(y_dev, (oof_prob >= t).astype(int), average="binary", zero_division=0)[2]
    THRESHOLD = round(float(max(np.linspace(0.05, 0.95, 91), key=lambda t: (round(f1_at(t), 6), -abs(t - 0.5)))), 2)
    print(f"Threshold chosen on out-of-fold scores: {THRESHOLD:.2f} (F1 {f1_at(THRESHOLD):.3f}). Your Settings value is ignored while AUTO_THRESHOLD = True.")
else:
    print(f"Using the threshold from Settings: {THRESHOLD}")

cv_scores = precision_recall_fscore_support(y_dev, (oof_prob >= THRESHOLD).astype(int), average="binary", zero_division=0)
print(f"Cross-validated estimate on the development set: precision {cv_scores[0]:.3f}, recall {cv_scores[1]:.3f}, F1 {cv_scores[2]:.3f}, "
      f"PR-AUC {average_precision_score(y_dev, oof_prob):.3f}  (a little optimistic, because the threshold was picked on these scores)")
cv_errors = dev.assign(prob=oof_prob.round(4), pred=(oof_prob >= THRESHOLD).astype(int))
cv_errors = cv_errors[cv_errors.pred != cv_errors.label][["text", "label", "prob", "source", "org_category", "is_hard_negative"]]
cv_errors.to_csv("outputs/errors_cross_validation.csv", index=False)
print(f"{len(cv_errors)} cross-validation mistakes saved to outputs/errors_cross_validation.csv (read these to decide what data to add next)")
print("Model trained on the full development set")

## 7. Simple keyword baselines (the model must beat these)
- Baseline A: flags any call containing a keyword like "police" or "bank".
- Baseline B: flags only when the caller states an identity ("this is ... from ...", "I am calling from ...").

In [ ]:
KEYWORDS = ["police", "officer", "bank", "ministry", "court", "immigration", "tax", "department", "investigation", "telco", "support"]
IDENTITY_RE = re.compile(r"\b(this is|i am|i'm|my name is)\b.{0,40}\b(from|with|calling from|of)\b|\bcalling from\b", re.I)

def baseline_a(texts):
    return np.array([int(any(k in t.lower() for k in KEYWORDS)) for t in texts])

def baseline_b(texts):
    return np.array([int(bool(IDENTITY_RE.search(t)) and any(k in t.lower() for k in KEYWORDS)) for t in texts])

## 8. Evaluate on the held-out test set
Prints precision, recall, F1, plus ranking and calibration scores, and saves charts into the `outputs` folder.

In [ ]:
def scores_for(y, pred, prob=None):
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    out = {"precision": p, "recall": r, "f1": f}
    if prob is not None:
        out.update({"roc_auc": roc_auc_score(y, prob), "pr_auc": average_precision_score(y, prob), "brier": brier_score_loss(y, prob)})
    return out

def evaluate(model, data, name):
    y = data["label"].values
    prob = model.predict_proba(data["text"])[:, 1]
    pred = (prob >= THRESHOLD).astype(int)
    table = pd.DataFrame({
        "keyword baseline A": scores_for(y, baseline_a(data["text"])),
        "identity baseline B": scores_for(y, baseline_b(data["text"])),
        "TF-IDF + LogReg": scores_for(y, pred, prob)}).T
    print(f"\n=== {name} (threshold {THRESHOLD}) ===")
    print(table.round(3).to_string())
    cm = confusion_matrix(y, pred, labels=[0, 1])
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax[0].text(j, i, str(v), ha="center", va="center", fontsize=14)
    ax[0].set_xticks([0, 1]); ax[0].set_yticks([0, 1])
    ax[0].set_xticklabels(["pred 0", "pred 1"]); ax[0].set_yticklabels(["true 0", "true 1"])
    ax[0].set_title(f"Confusion matrix: {name}")
    pr, rc, _ = precision_recall_curve(y, prob)
    ax[1].plot(rc, pr); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision-Recall curve")
    ax[1].set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(f"outputs/eval_{name.replace(' ', '_')}.png", dpi=150)
    plt.show()
    return prob, pred

test_prob, test_pred = evaluate(model, test, "test")

## 9. Deeper checks (these are what make the report credible)
1. Per-slice breakdown: does the model only work on one data source, one organisation type or one writing style?
2. Hard negatives: how often are legit official calls wrongly flagged?
3. Brand-masking test: does the score collapse if org names are hidden? (a sign the model memorised brands)
4. Coefficient audit: which words push the score up or down?
5. Reliability table: do scores of 0.8 really mean about 80% of the time? (the aggregator uses the score as a probability)
Mistakes are saved as CSV files in the `outputs` folder so you can read them.

In [ ]:
t = test.copy()
t["prob"] = test_prob
t["pred"] = test_pred

def slice_report(frame, name):
    """Recall on scam lines and false-alarm rate on genuine lines for each source, organisation type, style and hard-negative group."""
    out = []
    for col in ["source", "is_synthetic", "org_category", "style", "is_hard_negative"]:
        if col not in frame.columns or frame[col].nunique() < 2:
            continue
        for val, g in frame.groupby(col):
            pos, neg = g[g.label == 1], g[g.label == 0]
            out.append({"slice": col, "value": val, "n": len(g), "n_pos": len(pos), "n_neg": len(neg),
                        "recall_on_pos": (pos.pred == 1).mean() if len(pos) else np.nan,
                        "false_alarm_rate_on_neg": (neg.pred == 1).mean() if len(neg) else np.nan})
    rep = pd.DataFrame(out)
    if len(rep):
        rep.to_csv(f"outputs/slices_{name}.csv", index=False)
        print(rep.round(3).to_string(index=False))
        print("(slices with fewer than 10 rows are only a hint, not a result)")
    return rep

print("Per-slice results on the test set")
slice_report(t, "test")

hard = t[(t.label == 0) & (t.is_hard_negative == 1)]
if len(hard):
    print(f"\nHard negatives in test: {len(hard)}. Wrongly flagged: {(hard.pred == 1).mean():.1%}")

# Brand-masking robustness: compare a masked-training model and an unmasked-training model, both tested on brand-masked text
masked_test = text_utils.prep(test["text"], mask=True, caller=CALLER_ONLY)
dev_aug = with_noise(dev, RANDOM_STATE)
rob = []
for mask in (False, True):
    m = make_pipeline(best_c, mask, best_char).fit(dev_aug["text"], dev_aug["label"])
    f_plain = scores_for(test["label"].values, (m.predict_proba(test["text"])[:, 1] >= THRESHOLD).astype(int))["f1"]
    f_mask = scores_for(test["label"].values, (m.predict_proba(pd.Series(masked_test))[:, 1] >= THRESHOLD).astype(int))["f1"]
    rob.append({"trained_with_brand_masking": mask, "F1_normal_test": f_plain, "F1_test_with_brands_hidden": f_mask})
print("\nBrand-masking robustness\n", pd.DataFrame(rob).round(3).to_string(index=False))
print("If the unmasked model's F1 drops a lot when brands are hidden, it is relying on org names.")

# Coefficient audit
base_pipe = model.base if hasattr(model, "base") else model
names = np.array(base_pipe.named_steps["tfidf"].get_feature_names_out())
coefs = base_pipe.named_steps["clf"].coef_[0]
keep = np.array([not n.startswith("char__") for n in names])  # show word features only
names = np.array([n.replace("word__", "") for n in names[keep]])
coefs = coefs[keep]
order = np.argsort(coefs)
top_pos, top_neg = [str(x) for x in names[order[::-1][:20]]], [str(x) for x in names[order[:20]]]
print("\nTop 20 words/phrases pushing toward IMPERSONATION:\n", top_pos)
print("\nTop 20 words/phrases pushing toward NOT impersonation:\n", top_neg)
brand_tokens = set(text_utils.ORG_NAMES)
brand_hits = [w for w in top_pos if any(b in w.split() for b in brand_tokens)]
if len(brand_hits) >= 5:
    print("\nWARNING: many top features are organisation names:", brand_hits, "-> add more variety in org names.")
else:
    print("\nCoefficient audit OK: top features are not dominated by brand names.")
print("Check the NOT impersonation list too: words like hi, ready, tomorrow or appointment near the top mean the model learned a style shortcut.")

# Reliability table
bins = pd.cut(t["prob"], [0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
rel = t.groupby(bins).agg(n=("label", "size"), avg_score=("prob", "mean"), actual_rate=("label", "mean")).round(3)
print("\nReliability table on the test set (avg_score should be close to actual_rate)\n", rel.to_string())

# Errors to read by eye
errs = t[t.pred != t.label][["text", "label", "prob", "source", "org_category"]]
errs.to_csv("outputs/errors_test.csv", index=False)
print(f"\nMistakes on test: {len(errs)} (saved to outputs/errors_test.csv). First few:")
print(errs.head(8).to_string(index=False, max_colwidth=100))

# Speed check (the aggregator waits up to 2 seconds in total)
start = time.time()
for _ in range(200):
    model.predict_proba(["This is Officer Tan from the police, you are under investigation."])
print(f"\nAverage time per prediction: {(time.time() - start) / 200 * 1000:.1f} ms")

## 9b. How sure can we be? (confidence ranges, noisy speech, learning curve)
- **Confidence range:** the test set is small, so one number can mislead. We resample many times and report the range where the true score probably sits (95%).
  On the test set we resample whole script groups, so near-copies cannot make the range look tighter than it is.
- **Noisy speech:** the same test calls with dropped words, typos and "um" added, like speech-to-text would produce.
- **Learning curve:** inside each cross-validation fold, train on 25%, 50%, 75% and 100% of that fold's training scripts and score the fold's unseen rows. If the score is still climbing at 100%, more data will help. If it is flat, more data of the same kind will not.
  Scores here are never from rows the model was tuned on.

In [ ]:
def bootstrap_ci(y, prob, groups=None, n=BOOTSTRAPS, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y, prob = np.asarray(y), np.asarray(prob)
    pred = (prob >= THRESHOLD).astype(int)
    groups = np.arange(len(y)) if groups is None else np.asarray(groups)
    ug = np.unique(groups)
    members = {g: np.where(groups == g)[0] for g in ug}
    f1s, aps = [], []
    for _ in range(n):
        idx = np.concatenate([members[g] for g in rng.choice(ug, size=len(ug), replace=True)])
        if len(set(y[idx])) < 2:
            continue
        f1s.append(precision_recall_fscore_support(y[idx], pred[idx], average="binary", zero_division=0)[2])
        aps.append(average_precision_score(y[idx], prob[idx]))
    fmt = lambda a: f"{np.mean(a):.3f}  (95% range {np.percentile(a, 2.5):.3f} to {np.percentile(a, 97.5):.3f})"
    return {"F1": fmt(f1s), "PR_AUC": fmt(aps)}

ci = bootstrap_ci(test["label"].values, test_prob, groups=test_groups)
print("Test set confidence ranges:", ci)

noisy_test = noisy_copy(test, 999)
noisy_prob = model.predict_proba(noisy_test["text"])[:, 1]
f1_clean = scores_for(test["label"].values, (test_prob >= THRESHOLD).astype(int))["f1"]
f1_noisy = scores_for(test["label"].values, (noisy_prob >= THRESHOLD).astype(int))["f1"]
print(f"F1 on clean test calls: {f1_clean:.3f}   F1 on the same calls with speech-to-text style noise: {f1_noisy:.3f}")

lc = []
rng_lc = np.random.default_rng(RANDOM_STATE)
for frac in [0.25, 0.5, 0.75, 1.0]:
    for rep_i in range(2 if frac < 1 else 1):
        aps, f1s = [], []
        for tr, va in folds:
            tr_frame, tr_groups = dev.iloc[tr], dev_groups[tr]
            ug = np.unique(tr_groups)
            chosen = rng_lc.choice(ug, size=max(2, int(round(frac * len(ug)))), replace=False)
            sub = tr_frame[np.isin(tr_groups, chosen)]
            if sub["label"].nunique() < 2:
                continue
            m = fit_config(sub, best_c, best_char)
            pv = m.predict_proba(dev.iloc[va]["text"])[:, 1]
            aps.append(average_precision_score(y_dev[va], pv))
            f1s.append(precision_recall_fscore_support(y_dev[va], (pv >= 0.5).astype(int), average="binary", zero_division=0)[2])
        if aps:
            lc.append({"fraction_of_training_scripts": frac, "val_PR_AUC": np.mean(aps), "val_F1_at_0.5": np.mean(f1s)})
lcdf = pd.DataFrame(lc).groupby("fraction_of_training_scripts").mean().round(3)
print("\nLearning curve (cross-validated, unseen rows only)\n", lcdf.to_string())
plt.figure(figsize=(5, 3.5))
plt.plot(lcdf.index * 100, lcdf["val_PR_AUC"], marker="o", label="PR-AUC")
plt.plot(lcdf.index * 100, lcdf["val_F1_at_0.5"], marker="s", label="F1 (threshold 0.5)")
plt.xlabel("% of training scripts used"); plt.ylabel("score"); plt.ylim(0, 1.05); plt.legend(); plt.title("Learning curve")
plt.tight_layout(); plt.savefig("outputs/learning_curve.png", dpi=150); plt.show()

## 10. Gold test set (the final, untouched test)
This is the number to put in your report, with its confidence range. Keep every gold line out of training.
The `source` column separates lines written by AI models (`synthetic_independent`) from lines written or checked by people (for example `hand_written`).
Report them separately. The notebook also warns if a gold line is nearly identical to a training or test line.

In [ ]:
if gold is not None and len(gold):
    gold, gold_dropped = tidy(gold)
    if gold_dropped:
        print(f"Dropped {gold_dropped} gold rows whose label was not 0 or 1")
    gold["clean"] = text_utils.prep(gold["text"], mask=False, caller=CALLER_ONLY)
    leak = set(gold["clean"]) & set(df["clean"])
    if leak:
        print(f"WARNING: {len(leak)} gold examples also appear in the training data. Remove them from one side.")
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(pd.concat([df["clean"], gold["clean"]]))
    sim = (vec.transform(gold["clean"]) @ vec.transform(df["clean"]).T).max(axis=1).toarray().ravel()
    n_close = int((sim >= NEAR_DUP_THRESHOLD).sum())
    print(f"Highest similarity between a gold line and any training or test row: {sim.max():.2f}. Lines at or above {NEAR_DUP_THRESHOLD}: {n_close}")
    if n_close:
        print("WARNING: these gold lines are nearly identical to training lines, so the gold score would be too good. Remove or rewrite them.")
        print(gold.loc[sim >= NEAR_DUP_THRESHOLD, "text"].to_string(index=False, max_colwidth=100))
    print(f"Gold lines: {len(gold)} ({int(gold.label.sum())} impersonation). By source: {gold['source'].value_counts().to_dict()}")

    evaluate(model, gold, "gold")
    gp = model.predict_proba(gold["text"])[:, 1]
    g_eval = gold.assign(prob=gp, pred=(gp >= THRESHOLD).astype(int))
    print("Gold confidence ranges (all lines):", bootstrap_ci(gold["label"].values, gp))
    for s_name, g in g_eval.groupby("source"):
        if len(g) >= 20 and g["label"].nunique() == 2:
            print(f"  source = {s_name} ({len(g)} lines):", bootstrap_ci(g["label"].values, g["prob"].values))
    print("\nPer-slice results on the gold set")
    slice_report(g_eval, "gold")
    gold_errs = g_eval[g_eval.pred != g_eval.label][["text", "label", "prob", "source", "org_category"]]
    gold_errs.to_csv("outputs/errors_gold.csv", index=False)
    print(f"\nGold mistakes ({len(gold_errs)}, saved to outputs/errors_gold.csv). Do not tune the model to fix these lines, use them to decide what NEW data to write:")
    print(gold_errs.assign(prob=gold_errs.prob.round(3)).to_string(index=False, max_colwidth=100))
else:
    print(f"No gold data found. Add {GOLD_CSV} (columns text,label and ideally source) before trusting any numbers.")

## 11. Try it yourself
Edit the sentences and run this cell to see the score the aggregator will receive.

In [ ]:
def score_transcript(text):
    return float(model.predict_proba([text])[0, 1])

for s in ["Good afternoon, this is Inspector Lim from the police. Your account is linked to a case and you must cooperate.",
          "Hi, your parcel from the courier will arrive today between 2 and 4 pm.",
          "Hello this is customer service from your bank confirming your appointment tomorrow."]:
    print(f"{score_transcript(s):.3f}  <- {s}")

## 12. Package everything for Docker
Creates a `worker` folder with the model, helper file, FastAPI service, Dockerfile and requirements, then zips it for download.

In [ ]:
import sklearn, shutil
os.makedirs("worker", exist_ok=True)
joblib.dump(model, f"worker/{MODEL_VERSION}.joblib")
shutil.copy("text_utils.py", "worker/text_utils.py")

APP_PY = f'''
import joblib
from fastapi import FastAPI
from pydantic import BaseModel, Field
import text_utils  # must be importable so the saved model can find its cleaning functions

MODEL_FILE = "{MODEL_VERSION}.joblib"
model = joblib.load(MODEL_FILE)
app = FastAPI(title="Impersonation worker (English)")


class Req(BaseModel):
    text: str = Field(max_length=20000)  # oversized requests are rejected, never silently cut


@app.get("/healthz")
def healthz():
    return {{"status": "ok", "model": MODEL_FILE}}


@app.post("/predict")
def predict(req: Req):
    # Text is scored in memory only. It is never stored or logged.
    # Nothing is truncated: the whole text sent by the gateway is scored.
    score = float(model.predict_proba([req.text])[0, 1])
    return {{"impersonation_score": round(score, 4), "model_version": "{MODEL_VERSION}"}}
'''
with open("worker/app.py", "w") as f:
    f.write(APP_PY.lstrip())
with open("worker/requirements.txt", "w") as f:
    f.write(f"scikit-learn=={sklearn.__version__}\njoblib=={joblib.__version__}\nnumpy=={np.__version__}\nscipy\nfastapi\nuvicorn\n")
with open("worker/Dockerfile", "w") as f:
    f.write(f"""FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY text_utils.py app.py {MODEL_VERSION}.joblib ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
""")
nc_sources = sorted({x for x in df["source"].astype(str).unique() if any(m in x.lower() for m in NONCOMMERCIAL_SOURCE_MARKERS)})
card = {"model_version": MODEL_VERSION, "task": "English institutional impersonation (probability 0 to 1)",
        "features": f"TF-IDF ngram {NGRAM_RANGE}, min_df {MIN_DF}, max_features {MAX_FEATURES}",
        "brand_masking": MASK_BRANDS, "caller_only": CALLER_ONLY, "threshold_used_for_reporting": THRESHOLD, "C": best_c, "non_commercial_sources_in_training_data": nc_sources, "training_rows_by_source": {k: int(v) for k, v in dev["source"].value_counts().items()},
        "n_development_rows": int(len(dev)), "n_test_rows": int(len(test)), "cv_folds": N_FOLDS, "noisy_copies": NOISE_COPIES, "char_ngrams": best_char, "calibrated": CALIBRATE, "sklearn_version": sklearn.__version__,
        "test_metrics": scores_for(test["label"].values, test_pred, test_prob)}
with open("worker/model_card.json", "w") as f:
    json.dump(card, f, indent=2, default=float)
shutil.make_archive("impersonation_worker", "zip", "worker")
print("Created impersonation_worker.zip with:", sorted(os.listdir("worker")))
if nc_sources:
    print("WARNING: the training data includes non-commercial sources:", nc_sources, "-> check the licence before this model is shipped.")
if df["is_synthetic"].all():
    print("REMINDER: every training row is synthetic. Report results as synthetic.")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("impersonation_worker.zip")
    for f in os.listdir("outputs"):
        files.download(os.path.join("outputs", f))

## Optional Part B: helpers for turning raw call transcripts into windows
These turn full conversations into short windows of **caller words only**, ready for you to review by hand.
Every caller turn lands in at least one window, including the last turns of the call (nothing is cut off). Windows overlap so a claim on the edge of one window is still seen whole in the next.
Nothing here labels data automatically. A regex flag is only a hint for your manual review.

In [ ]:
def caller_units(raw):
    """Every caller turn, in order. With no speaker labels, the text is cut into sentences instead so long transcripts are still fully covered."""
    raw = str(raw)
    if text_utils.LABEL_SPLIT_RE.search(raw):
        parts = text_utils.LABEL_SPLIT_RE.split(raw)
        units = [parts[i + 1].strip() for i in range(1, len(parts), 2) if parts[i].lower() in text_utils.REMOTE_LABELS]
        if parts[0].strip():
            units.insert(0, parts[0].strip())
    else:
        units = re.split(r"(?<=[.?!])\s+", raw.strip())
    return [u for u in units if u]

def make_windows(raw, turns=WINDOW_TURNS, stride=2):
    """Overlapping windows of `turns` caller turns. The final window always ends at the last turn, so the tail is never dropped."""
    assert 1 <= stride <= turns, "stride must be between 1 and turns, or turns would fall in gaps"
    units = caller_units(raw)
    if not units:
        return []
    if len(units) <= turns:
        return [" ".join(units)]
    starts = list(range(0, len(units) - turns + 1, stride))
    if starts[-1] + turns < len(units):
        starts.append(len(units) - turns)
    return [" ".join(units[i:i + turns]) for i in starts]

def check_coverage(raw, turns=WINDOW_TURNS, stride=2):
    """True if every caller turn appears in at least one window."""
    units = caller_units(raw)
    windows = make_windows(raw, turns, stride)
    return all(any(u in w for w in windows) for u in units)

demo_call = ("Caller: Hello, this is Officer Tan from the police. Receiver: Who? Caller: There is a case under your name. "
             "Receiver: What case? Caller: Please stay on the line. Receiver: Okay. Caller: I need to verify your identity. "
             "Receiver: How? Caller: First confirm your full name. Receiver: Why? Caller: This is a confidential investigation.")
for i, w in enumerate(make_windows(demo_call), 1):
    print(f"Window {i}: {w}")
print("Every caller turn covered:", check_coverage(demo_call))
assert all(check_coverage("Caller: " + " Receiver: ok. Caller: ".join(f"turn {k}." for k in range(n))) for n in range(1, 25))
print("Coverage test passed for calls of 1 to 24 caller turns")

In [ ]:
# Optional: window your own raw transcripts and write candidate rows to review_candidates.csv
# Put a file called raw_calls.csv next to this notebook with columns: call_id, text (and optionally source).
# Then label each row 1, 0 or U (unclear) by hand with the labelling rule. U rows are dropped when the data is loaded.
RAW_CALLS_CSV = "raw_calls.csv"
CANDIDATE_RE = re.compile(r"\b(this is|i am|calling from|i'm with)\b.{0,60}\b(police|bank|ministry|department|officer|support|court|tax|immigration)\b", re.I)
if os.path.exists(RAW_CALLS_CSV):
    raw = pd.read_csv(RAW_CALLS_CSV)
    out = []
    for _, r in raw.iterrows():
        for j, w in enumerate(make_windows(r["text"])):
            out.append({"text": w, "label": "", "source": r.get("source", "raw_calls"), "scenario_id": f"call_{r['call_id']}",
                        "window_index": j, "regex_hint": int(bool(CANDIDATE_RE.search(w)))})
    pd.DataFrame(out).to_csv("review_candidates.csv", index=False)
    print("Wrote review_candidates.csv with", len(out), "windows. Label them by hand.")
else:
    print(f"No {RAW_CALLS_CSV} found, so nothing was written. This step is optional.")